# Notebook 02: Data Cleaning and Canonical Model

**Project:** Global Layoffs Intelligence (Python + PostgreSQL + Power BI + ML)
**Phase:** 2 of 6

This notebook turns the two raw files into **one canonical table**, `data/processed/layoff_events.csv`, with **one row per source record**.
It only applies the rules you approved (A-01 to A-16 and P1 to P5 in `reports/`).

Ground rules followed in every step:

| Rule | How |
|---|---|
| Raw files are never changed | SHA-256 is checked at the start and the end |
| Original values are kept | every changed field keeps a `*_raw` column next to the cleaned one |
| Missing stays missing | no missing value is filled with 0, `False` or `"No"` |
| Nothing is deleted or merged | 5,035 rows in, 5,035 rows out; suspected duplicates are *linked*, never merged |
| Every change is logged | `transformation_log.csv` lists each rule, the rows it changed and examples |
| The tech file stays separate | `source_confidence = low` on every tech row |

**How to run:** same kernel as notebook 01 (*Python (layoffs-env)*, see notebook 01 for setup). Run notebook 01 first, because this notebook reads its outputs from `data/interim/`. Then *Run All*.

## 1. Setup

**Objective.** Load libraries, find the project folders and import the cleaning rules.

**Concept.** The rules live in `src/phase2_cleaning.py` and `src/phase2_mappings.py`, so the *same* code is run here and by the automated tests in `tests/`.
This notebook calls them one step at a time and explains each step.

In [1]:
from pathlib import Path
import sys
import hashlib
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)


def find_project_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "data" / "raw").is_dir():
            return folder
    raise FileNotFoundError("Start Jupyter inside the project folder.")


PROJECT_ROOT  = find_project_root(Path.cwd())
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR   = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AUDIT_DIR     = PROJECT_ROOT / "reports" / "audit"
FIG_DIR       = PROJECT_ROOT / "reports" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GLOBAL_FILE = RAW_DIR / "layoffs_2019_2026_sample.csv"
TECH_FILE   = RAW_DIR / "tech_layoffs_2025_2026.csv"

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import phase2_cleaning as pc       # Phase 2 rules
import phase2_mappings as pm       # company + industry mappings (A-02, A-04)
print("Project root:", PROJECT_ROOT)

Project root: /home/claude/global-layoffs-intelligence


**Two logs.** `change_log` records every transformation (written to `data/processed/transformation_log.csv`).
`check_log` records every validation check with its real result (appended to the project audit history).

In [2]:
change_log = []
check_log = []


def check(name, passed, detail=""):
    # record a validation result and stop the notebook if it failed
    check_log.append({"checked_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
                      "check": name, "passed": bool(passed), "detail": str(detail)})
    print(("PASS " if passed else "FAIL ") + name + (f"  ({detail})" if detail else ""))
    assert passed, name

## 2. Raw files are unchanged

**Objective.** Confirm the raw files are byte-for-byte the files Phase 1 fingerprinted.

**Concept.** We recompute SHA-256 and compare it with `reports/audit/raw_file_manifest.csv` from Phase 1. If anything differs, the notebook stops.

In [3]:
def sha256_of(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


manifest = pd.read_csv(AUDIT_DIR / "raw_file_manifest.csv")
manifest["sha256_now"] = [sha256_of(RAW_DIR / f) for f in manifest["file"]]
check("raw files match Phase 1 SHA-256", (manifest["sha256"] == manifest["sha256_now"]).all(),
      ", ".join(f"{f}={h[:8]}" for f, h in zip(manifest["file"], manifest["sha256_now"])))
manifest[["file", "size_kb"]]

PASS raw files match Phase 1 SHA-256  (SOURCE_DATA_CARD.md=8e93c93c, layoffs_2019_2026_sample.csv=a181c6ce, tech_layoffs_2025_2026.csv=f011a4d0)


,file,size_kb
0,SOURCE_DATA_CARD.md,102.0
1,layoffs_2019_2026_sample.csv,2126.7
2,tech_layoffs_2025_2026.csv,4.8


## 3. Load the raw data and the Phase 1 outputs

**Objective.** Read the two raw files (for original values), the Phase 1 staged files (for quality flags) and the two duplicate-review files.

**Concept.** We read the raw files exactly as Phase 1 did (`keep_default_na=False`, only empty cells are missing).
The Phase 1 record IDs (`G-00001`, `T-00001`) are rebuilt from row order and checked against the staged files, so every row keeps its link to its source line.

In [4]:
raw_global = pd.read_csv(GLOBAL_FILE, keep_default_na=False, na_values=[""])
raw_tech   = pd.read_csv(TECH_FILE, keep_default_na=False, na_values=[""])

staged_global = pd.read_csv(INTERIM_DIR / "global_layoffs_staged.csv", keep_default_na=False, na_values=[""])
staged_tech   = pd.read_csv(INTERIM_DIR / "tech_layoffs_staged.csv", keep_default_na=False, na_values=[""])
kept_pairs     = pd.read_csv(INTERIM_DIR / "possible_duplicate_pairs.csv")
excluded_pairs = pd.read_csv(INTERIM_DIR / "excluded_candidate_pairs.csv")

pd.DataFrame({"input": ["raw global", "raw tech", "staged global", "staged tech", "kept pairs", "excluded pairs"],
              "rows": [len(raw_global), len(raw_tech), len(staged_global), len(staged_tech),
                       len(kept_pairs), len(excluded_pairs)]})

,input,rows
0,raw global,5000
1,raw tech,35
2,staged global,5000
3,staged tech,35
4,kept pairs,18
5,excluded pairs,97


In [5]:
check("raw row counts are 5,000 + 35", (len(raw_global), len(raw_tech)) == (5000, 35))
check("staged files hold the same rows as raw", (len(staged_global), len(staged_tech)) == (5000, 35))
check("staged raw columns equal raw values",
      staged_global[raw_global.columns].equals(raw_global) and staged_tech[raw_tech.columns].equals(raw_tech))
check("duplicate-review files have 18 + 97 pairs", (len(kept_pairs), len(excluded_pairs)) == (18, 97))

PASS raw row counts are 5,000 + 35
PASS staged files hold the same rows as raw
PASS staged raw columns equal raw values
PASS duplicate-review files have 18 + 97 pairs


## 4. Step 1: map source columns to canonical names (no values changed)

**Objective.** Put both sources into the same column layout. This step only **renames and copies**. No value is changed.

**Concept: canonical model.** "Canonical" means *the one agreed shape*. Each source column is copied into its canonical column.
Where a source has no such column the cell is missing (for example, global rows have no `severance_weeks`). It is **not** 0.
The two dictionaries below are also saved as `field_lineage.csv`, so anyone can see where every column came from.

In [6]:
# canonical column -> source column (None = this source has no such field -> missing)
GLOBAL_TO_CANONICAL = {
    "source_event_id": "layoff_id", "company_name_raw": "company_name", "company_domain": "company_domain",
    "event_date_raw": "date", "industry_raw": "sector", "country_raw": "country", "hq_country_raw": "company_country",
    "location_raw": None, "city": "company_city", "state": "company_state",
    "employees_laid_off": "headcount_cut", "headcount_status": "headcount_status", "workforce_pct": "pct_workforce",
    "total_employees": None, "reason": None, "department": None, "severance_weeks": None, "ai_related": None,
    "is_furlough": "is_furlough", "news_source": "news_source", "news_url_raw": "news_url",
    "news_excerpt": "news_excerpt", "news_mentions": "news_mentions", "source_profile_url": "fliar_profile_url",
}
TECH_TO_CANONICAL = {
    "source_event_id": None, "company_name_raw": "Company", "company_domain": None,
    "event_date_raw": "Date", "industry_raw": "Industry", "country_raw": "Country", "hq_country_raw": None,
    "location_raw": "Location", "city": None, "state": None,
    "employees_laid_off": "Employees_Laid_Off", "headcount_status": None, "workforce_pct": "Percentage_Workforce",
    "total_employees": "Total_Employees", "reason": "Reason", "department": "Department",
    "severance_weeks": "Severance_Weeks", "ai_related": "AI_Related",
    "is_furlough": None, "news_source": None, "news_url_raw": None, "news_excerpt": None, "news_mentions": None,
    "source_profile_url": None,
}
# Source columns intentionally NOT copied (derived from the date; Phase 1 found 100% agreement)
DROPPED = {"global": ["year", "month"], "tech": ["Year", "Month", "Month_Name", "Quarter"]}

In [7]:
def base_frame(raw, mapping, prefix, dataset, confidence, file_name):
    out = pd.DataFrame(index=raw.index)
    out["event_key"] = [f"{prefix}-{i:05d}" for i in range(1, len(raw) + 1)]     # same IDs as Phase 1
    out["source_dataset"] = dataset
    out["source_confidence"] = confidence
    out["source_file"] = file_name
    out["source_row_number"] = np.arange(2, len(raw) + 2)                       # CSV line (header = 1)
    for canonical, source_col in mapping.items():
        out[canonical] = raw[source_col] if source_col else pd.NA
    return out


g = base_frame(raw_global, GLOBAL_TO_CANONICAL, "G", pc.GLOBAL_NAME, "high", GLOBAL_FILE.name)
t = base_frame(raw_tech, TECH_TO_CANONICAL, "T", pc.TECH_NAME, "low", TECH_FILE.name)

check("event_key matches Phase 1 record_id (global)", g["event_key"].tolist() == staged_global["record_id"].tolist())
check("event_key matches Phase 1 record_id (tech)", t["event_key"].tolist() == staged_tech["record_id"].tolist())
check("every source column is mapped or listed as dropped",
      set(raw_global.columns) == {v for v in GLOBAL_TO_CANONICAL.values() if v} | set(DROPPED["global"]) and
      set(raw_tech.columns) == {v for v in TECH_TO_CANONICAL.values() if v} | set(DROPPED["tech"]))

events = pd.concat([g, t], ignore_index=True)
print("combined rows:", len(events))
events.head(3)

PASS event_key matches Phase 1 record_id (global)
PASS event_key matches Phase 1 record_id (tech)
PASS every source column is mapped or listed as dropped
combined rows: 5035


,event_key,source_dataset,source_confidence,source_file,source_row_number,source_event_id,company_name_raw,company_domain,event_date_raw,industry_raw,country_raw,hq_country_raw,location_raw,city,state,employees_laid_off,headcount_status,workforce_pct,total_employees,reason,department,severance_weeks,ai_related,is_furlough,news_source,news_url_raw,news_excerpt,news_mentions,source_profile_url
0,G-00001,fliar_global_sample,high,layoffs_2019_2026_sample.csv,2,3eabf5c11805,Popular's integration into Santander,NaN,2019-01-04,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,not_disclosed,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,False,NaN,NaN,"Popular's integration into Santander, CaixaBan...",1,NaN
1,G-00002,fliar_global_sample,high,layoffs_2019_2026_sample.csv,3,964aaab244e9,Selecta Biosciences,selectabio.com,2019-01-04,Life Sciences,United States,United States,<NA>,Watertown,New York,NaN,not_disclosed,36.0,<NA>,<NA>,<NA>,<NA>,<NA>,False,biospace.com,https://www.biospace.com/selecta-biosciences-n...,"Selecta Biosciences, based in Watertown, Mass....",1,https://fliarbi.com/company/selecta-bioscience...
2,G-00003,fliar_global_sample,high,layoffs_2019_2026_sample.csv,4,b8fd1f9ace14,Avianca,avianca.com,2019-01-05,Travel and Hospitality,Colombia,Colombia,<NA>,Bogota,NaN,NaN,failed_check,2.5,<NA>,<NA>,<NA>,<NA>,<NA>,False,em.com.br,https://www.em.com.br/app/noticia/economia/201...,"Avianca, an airline that owns 13% of the domes...",2,https://fliarbi.com/company/avianca/1nbdp96dct...


In [8]:
# Validation: the dropped year/month columns really agree with the date (so dropping them loses nothing)
gd, td = pd.to_datetime(raw_global["date"]), pd.to_datetime(raw_tech["Date"])
check("dropped global year/month agree with date",
      (raw_global["year"] == gd.dt.year).all() and (raw_global["month"] == gd.dt.strftime("%Y-%m")).all())
check("dropped tech Year/Month/Month_Name/Quarter agree with Date",
      (raw_tech["Year"] == td.dt.year).all() and (raw_tech["Month"] == td.dt.month).all()
      and (raw_tech["Month_Name"] == td.dt.month_name()).all() and (raw_tech["Quarter"] == td.dt.quarter).all())

PASS dropped global year/month agree with date
PASS dropped tech Year/Month/Month_Name/Quarter agree with Date


## 5. Step 2: company names (A-02, P2)

**Objective.** Create a clean `company_name` while keeping `company_name_raw` exactly as it was.

**Concept.** Three kinds of change, all from `src/phase2_mappings.py`:

1. **Rule:** trim spaces, straighten curly apostrophes (’ → '), drop a legal suffix at the very end (`Apple Inc` → `Apple`). Brand words such as *Company* and *Group* are kept.
2. **Confident manual mapping:** for sentence-like names, tickers and aliases (`Electric-car maker Tesla Inc. TSLA` → `Tesla`, `X (Twitter)` → `Twitter`).
3. **P2, kept and flagged:** the 13 uncertain names (subsidiary vs parent, business units, descriptions) keep their **exact raw name**, with `company_name_review_status = needs_review`. No subsidiary is merged into a parent.

Then **case harmonisation**: names that differ only by capital letters (`SNAP` / `Snap`) take one spelling: a spelling that is **not all capitals** is preferred
(so `Snap`, not `SNAP`), then the one used by the most rows, then alphabetical order.
Rows marked `needs_review` are left out of this step.

In [9]:
mapped = events["company_name_raw"].map(pm.clean_company)        # (name, method, review_status, note)
events["company_name"] = mapped.str[0]
events["company_name_method"] = mapped.str[1]
events["company_name_review_status"] = mapped.str[2]

pc.log_change(change_log, "T01", "A-02/P2", "company_name", events["company_name_raw"], events["company_name"],
              "legal-suffix/spacing rule + confident manual mappings; uncertain names kept raw", events["event_key"])
events["company_name_method"].value_counts()

company_name_method
unchanged                  4798
rule_suffix_or_spacing      198
manual_mapping               26
kept_raw_pending_review      13
Name: count, dtype: int64

In [10]:
before_case = events["company_name"].copy()
eligible = events["company_name_review_status"] != "needs_review"
spelling_rows = (events[eligible].groupby(["company_name"]).size().rename("rows").reset_index())
spelling_rows["key"] = spelling_rows["company_name"].str.lower()
spelling_rows["all_caps"] = spelling_rows["company_name"].str.isupper()
spelling_rows = spelling_rows.sort_values(["key", "all_caps", "rows", "company_name"], ascending=[True, True, False, True])
preferred = spelling_rows.drop_duplicates("key").set_index("key")["company_name"]

events.loc[eligible, "company_name"] = events.loc[eligible, "company_name"].str.lower().map(preferred)
case_changed = pc.changed_mask(before_case, events["company_name"])
events.loc[case_changed, "company_name_method"] = events.loc[case_changed, "company_name_method"] + "+case_harmonised"
n = pc.log_change(change_log, "T01b", "A-02", "company_name", before_case, events["company_name"],
                  "spellings differing only by capital letters: prefer not-all-caps, then most rows", events["event_key"])
print("rows whose spelling was harmonised:", n)
events.loc[case_changed, ["event_key", "company_name_raw", "company_name"]]

rows whose spelling was harmonised: 7


,event_key,company_name_raw,company_name
2007,G-02008,SNAP,Snap
2061,G-02062,Cargologic Germany,CargoLogic Germany
2084,G-02085,SNAP,Snap
2299,G-02300,SNAP,Snap
3215,G-03216,SNAP,Snap
4045,G-04046,DON'T NOD,Don't Nod
4902,G-04903,SNAP,Snap


In [11]:
check("company_name_raw is unchanged", events["company_name_raw"].tolist() ==
      raw_global["company_name"].tolist() + raw_tech["Company"].tolist())
review = events["company_name_review_status"] == "needs_review"
check("P2: every needs_review name equals its raw name", (events.loc[review, "company_name"] == events.loc[review, "company_name_raw"]).all(),
      f"{review.sum()} rows")
check("no empty company names", events["company_name"].notna().all() and (events["company_name"].str.len() > 0).all())

PASS company_name_raw is unchanged
PASS P2: every needs_review name equals its raw name  (13 rows)
PASS no empty company names


## 6. Step 3: dates (parse, then flag the 6 suspect tech dates; A-07)

**Objective.** Turn the date text into a real date, add year/quarter/month, and flag the tech rows whose date looks wrong. **No date is replaced.**

**Concept.** Parsing uses `parse_dates` from Phase 1 (`src/phase1_validation.py`), the same rule that passed its tests.
For A-07 a tech row is `suspect_date` when its headcount is within 5% of a *global* event that is more than 120 days away (evidence from the Phase 1 review file).
The reason, including the pair ID, is written into `date_flag_reason` so a reviewer can follow it.

In [12]:
events["event_date"] = pc.parse_dates(events["event_date_raw"])
events["event_year"] = events["event_date"].dt.year.astype("Int64")
events["event_quarter"] = events["event_date"].dt.quarter.astype("Int64")
events["event_month"] = events["event_date"].dt.month.astype("Int64")
pc.log_change(change_log, "T02", "mechanical", "event_date", events["event_date_raw"],
              events["event_date"].dt.strftime("%Y-%m-%d"), "text parsed to date (format unchanged)", events["event_key"])
check("all dates parse", events["event_date"].notna().all())

PASS all dates parse


In [13]:
suspect = pc.suspect_tech_dates(kept_pairs)
events["date_reliability"] = "as_reported"
events["date_flag_reason"] = pd.Series(pd.NA, index=events.index, dtype="string")
is_suspect = events["event_key"].isin(suspect["event_key"])
events.loc[is_suspect, "date_reliability"] = "suspect_date"
events.loc[is_suspect, "date_flag_reason"] = events.loc[is_suspect, "event_key"].map(
    suspect.set_index("event_key")["date_flag_reason"])
change_log.append({"rule_id": "T03", "decision": "A-07", "column": "date_reliability",
                   "description": "flag only: tech rows matching a global event's size >120 days away; dates unchanged",
                   "rows_changed": int(is_suspect.sum()), "examples": " | ".join(suspect["event_key"])})
events.loc[is_suspect, ["event_key", "company_name", "event_date_raw", "date_flag_reason"]]

,event_key,company_name,event_date_raw,date_flag_reason
5001,T-00002,Twitch,2025-01-09,headcount 500 within 0.0% of global G-03089 (5...
5006,T-00007,Dell,2025-02-05,headcount 6650 within 1.5% of global G-02557 (...
5012,T-00013,Roku,2025-03-30,headcount 360 within 0.0% of global G-02887 (3...
5013,T-00014,Accenture,2025-03-30,headcount 19000 within 0.0% of global G-02252 ...
5017,T-00018,Unity,2025-05-08,headcount 1800 within 0.0% of global G-03142 (...
5030,T-00031,Adobe,2026-01-28,headcount 680 within 0.0% of global G-02463 (6...


In [14]:
check("A-07: exactly 6 suspect dates", is_suspect.sum() == 6)
check("A-07: the 6 are Twitch, Dell, Roku, Accenture, Unity, Adobe",
      set(events.loc[is_suspect, "company_name"]) == {"Twitch", "Dell", "Roku", "Accenture", "Unity", "Adobe"})
check("A-07: suspect dates keep their original value",
      (events.loc[is_suspect, "event_date"].dt.strftime("%Y-%m-%d") == events.loc[is_suspect, "event_date_raw"]).all())

PASS A-07: exactly 6 suspect dates
PASS A-07: the 6 are Twitch, Dell, Roku, Accenture, Unity, Adobe
PASS A-07: suspect dates keep their original value


## 7. Step 4: industry groups (A-04, P1, P3, P5)

**Objective.** Map 49 global sector labels and 27 tech industry labels to **16 shared groups + `Unknown`**.

**Concept.**
- **P1:** the global label *High Tech* (a mix of OpenAI, Meta, Google, iRobot, DJI...) gets its own group, *Technology (mixed)*.
- **P3:** each row keeps **its own source label**. When the two sources disagree for the same company (e.g. Dell is *Hardware* in the tech file but *Investment Industry* in the global file),
  we **record** it in `cross_source_industry_disagreements.csv` and flag it, but never override by company.
- **P5:** tech *E-commerce* → *Retail & Consumer* also covers Shopify. That limitation is documented and there is no company override.

In [15]:
events["industry_group"] = [pm.industry_group(label, "global" if ds == pc.GLOBAL_NAME else "tech")
                            for label, ds in zip(events["industry_raw"], events["source_dataset"])]
pc.log_change(change_log, "T04", "A-04/P1", "industry_group", events["industry_raw"], events["industry_group"],
              "label-level mapping to 16 groups + Unknown (missing sector -> Unknown)", events["event_key"])
check("every row has a known industry group", events["industry_group"].isin(pm.INDUSTRY_GROUPS).all())
check("P1: High Tech rows -> Technology (mixed)",
      (events.loc[events["industry_raw"] == "High Tech", "industry_group"] == "Technology (mixed)").all(),
      f"{(events['industry_raw'] == 'High Tech').sum()} rows")
events.groupby(["industry_group", "source_dataset"]).size().unstack(fill_value=0)   # counts of rows, so 0 is correct

PASS every row has a known industry group
PASS P1: High Tech rows -> Technology (mixed)  (70 rows)


source_dataset,fliar_global_sample,tech_layoffs_2025_2026
industry_group,,
Automotive,185,2
Business Services,184,1
Education & Public Sector,65,1
Energy & Resources,191,0
Finance & Fintech,375,3
Food & Agriculture,268,0
Hardware & Semiconductors,49,4
Healthcare & Life Sciences,298,0
Industrial & Manufacturing,208,0


In [16]:
# P3: same company (exact clean name) in both sources but a different industry group -> record, don't override
tech_ind = events[events["source_dataset"] == pc.TECH_NAME][["event_key", "company_name", "industry_raw", "industry_group"]]
glob_ind = events[events["source_dataset"] == pc.GLOBAL_NAME][["event_key", "company_name", "industry_raw", "industry_group"]]
both = tech_ind.merge(glob_ind, on="company_name", suffixes=("_tech", "_global"))
both["groups_agree"] = both["industry_group_tech"] == both["industry_group_global"]
disagree = (both[~both["groups_agree"]]
            .groupby(["company_name", "event_key_tech", "industry_raw_tech", "industry_group_tech",
                      "industry_raw_global", "industry_group_global"], as_index=False)
            .agg(global_event_keys=("event_key_global", lambda s: ";".join(sorted(s))), global_rows=("event_key_global", "count")))
disagree.to_csv(PROCESSED_DIR / "cross_source_industry_disagreements.csv", index=False)

conflict_keys = set(disagree["event_key_tech"]) | {k for ks in disagree["global_event_keys"] for k in ks.split(";")}
events["qc_industry_cross_source_conflict"] = events["event_key"].isin(conflict_keys)
print("companies with a cross-source industry disagreement:", disagree["company_name"].nunique(),
      "| rows flagged:", events["qc_industry_cross_source_conflict"].sum())
disagree[["company_name", "industry_raw_tech", "industry_group_tech", "industry_raw_global", "industry_group_global", "global_rows"]]

companies with a cross-source industry disagreement: 9 | rows flagged: 46


,company_name,industry_raw_tech,industry_group_tech,industry_raw_global,industry_group_global,global_rows
0,Accenture,Consulting,Business Services,Investment Industry,Finance & Fintech,11
1,Dell,Hardware,Hardware & Semiconductors,Investment Industry,Finance & Fintech,2
2,Google,Search/AI,Software & IT,High Tech,Technology (mixed),1
3,Lyft,Ridesharing,Travel & Transport,Consumer,Retail & Consumer,6
4,Meta,Social Media,Software & IT,High Tech,Technology (mixed),1
5,Rivian,Automotive/EV,Automotive,Energy Tech,Energy & Resources,2
6,Snap,Social Media,Software & IT,Consumer,Retail & Consumer,5
7,Twitch,Live Streaming,Media & Gaming,Consumer,Retail & Consumer,3
8,Unity,Gaming/Software,Media & Gaming,Financial Services,Finance & Fintech,6


In [17]:
check("P3: industry_raw unchanged (no company override)",
      events["industry_raw"].astype("string").fillna("<NA>").tolist() ==
      raw_global["sector"].astype("string").fillna("<NA>").tolist() + raw_tech["Industry"].astype("string").tolist())

PASS P3: industry_raw unchanged (no company override)


## 8. Step 5: countries, region and location (A-03, A-05, A-05b, A-06)

**Objective.** Standardise country labels, keep event country and HQ country separate, add a region, and split the tech `Location`.

**Concept.**
- **A-03:** only four relabels are approved: `USA` and `United States (usa)` → `United States`, `United Kingdom (uk)` → `United Kingdom`, `United States (cnmi)` → `Northern Mariana Islands`.
- **A-05:** global `country` = where the layoff happened; `hq_country` = the company's HQ. One is **never** used to fill the other.
- **A-05b:** tech `Country` looks like the HQ, so `country_basis = hq_assumed` for tech rows.
- **New finding:** a few global values are US states or fragments (`Arizona`, `New York`, `Quebec`, `Au`, `Sa`...). There is no approved rule for them, so they are **kept as-is** and flagged `qc_country_not_a_country`, with region `Unresolved`.

In [18]:
events["country"] = pc.standardise_country(events["country_raw"])
events["hq_country"] = pc.standardise_country(events["hq_country_raw"])
events["country_basis"] = np.where(events["source_dataset"] == pc.GLOBAL_NAME, "event_location", "hq_assumed")
events["region"] = pc.region_of(events["country"])
events["qc_country_not_a_country"] = (events["country"].isin(pc.NOT_A_COUNTRY) |
                                      events["hq_country"].isin(pc.NOT_A_COUNTRY))
pc.log_change(change_log, "T05", "A-03", "country", events["country_raw"], events["country"],
              "4 approved relabels + trim", events["event_key"])
pc.log_change(change_log, "T05b", "A-03/A-05", "hq_country", events["hq_country_raw"], events["hq_country"],
              "4 approved relabels + trim; never filled from country", events["event_key"])
events["region"].value_counts()

region
North America                2161
Europe                       1446
Asia Pacific                  680
Unknown                       470
Middle East & Africa          168
Latin America & Caribbean     104
Unresolved                      6
Name: count, dtype: Int64

In [19]:
loc = pc.split_location(events["location_raw"])
is_tech = events["source_dataset"] == pc.TECH_NAME
city_before, state_before = events["city"].copy(), events["state"].copy()
events.loc[is_tech, "city"] = loc.loc[is_tech, "city"]
events.loc[is_tech, "state"] = loc.loc[is_tech, "state"]
pc.log_change(change_log, "T06", "A-06", "city", city_before, events["city"], "tech: text before the comma", events["event_key"])
pc.log_change(change_log, "T06b", "A-06", "state", state_before, events["state"],
              "tech: after the comma only if a US state code", events["event_key"])
events.loc[is_tech, ["event_key", "location_raw", "city", "state"]].drop_duplicates("location_raw").head(8)

,event_key,location_raw,city,state
5000,T-00001,"Pittsburgh, PA",Pittsburgh,PA
5001,T-00002,"San Francisco, CA",San Francisco,CA
5004,T-00005,"Boston, MA",Boston,MA
5005,T-00006,"San Jose, CA",San Jose,CA
5006,T-00007,"Round Rock, TX",Round Rock,TX
5009,T-00010,"Irvine, CA",Irvine,CA
5011,T-00012,"Armonk, NY",Armonk,NY
5013,T-00014,"Dublin, Ireland",Dublin,<NA>


In [20]:
check("A-05: missing event country not filled from HQ",
      events["country"].isna().sum() == events["country_raw"].isna().sum(), f"{events['country'].isna().sum()} missing")
check("A-05: hq_country missing count unchanged", events["hq_country"].isna().sum() == events["hq_country_raw"].isna().sum())
check("no country left in an unapproved bracket form", not events["country"].str.contains(r"\(", na=False).any())
check("region is never 'Other' (every real country mapped)", (events["region"] != "Other").all(),
      events["region"].value_counts().to_dict())
flagged = events[events["qc_country_not_a_country"]]
print("rows flagged:", len(flagged))
flagged[["event_key", "company_name", "country_raw", "hq_country_raw", "region"]]

PASS A-05: missing event country not filled from HQ  (470 missing)
PASS A-05: hq_country missing count unchanged
PASS no country left in an unapproved bracket form
PASS region is never 'Other' (every real country mapped)  ({'North America': 2161, 'Europe': 1446, 'Asia Pacific': 680, 'Unknown': 470, 'Middle East & Africa': 168, 'Latin America & Caribbean': 104, 'Unresolved': 6})
rows flagged: 7


,event_key,company_name,country_raw,hq_country_raw,region
112,G-00113,Minerals Council South Africa,Sa,Sa,Unresolved
900,G-00901,HF Coors Ceramics Factory Store,Arizona,NaN,Unresolved
2783,G-02784,Aruze Gaming,Nevada,United States,Unresolved
3255,G-03256,Melbourne Rebels,Australia,Au,Asia Pacific
4266,G-04267,VERTEX,Rhode Island,United States,Unresolved
4402,G-04403,Genesis,New York,United States,Unresolved
4954,G-04955,Furniture South Shore,Quebec,NaN,Unresolved


## 9. Step 6: numbers and headcount reliability (A-08, A-09)

**Objective.** Store counts as whole numbers that can be missing (`Int64`), label how reliable each headcount is, and prove no missing value became 0.

**Concept.** Pandas reads a column with gaps as decimals (`1200.0`). `Int64` (capital I) is a whole-number type that **allows missing values**.
We only convert after checking every filled value really is a whole number. Tech headcounts get `headcount_status = reported_unverified` (A-08).
`total_employees` stays missing for global rows (A-09: not derived).

In [21]:
for col in ["employees_laid_off", "total_employees", "severance_weeks", "news_mentions"]:
    values = pd.to_numeric(events[col])
    check(f"{col}: all filled values are whole numbers", ((values.dropna() % 1) == 0).all())
    events[col] = values.astype("Int64")
events["workforce_pct"] = pd.to_numeric(events["workforce_pct"]).astype("Float64")

status_before = events["headcount_status"].copy()
events.loc[is_tech, "headcount_status"] = "reported_unverified"
pc.log_change(change_log, "T07", "A-08", "headcount_status", status_before, events["headcount_status"],
              "tech rows labelled reported_unverified", events["event_key"])
events.groupby("source_dataset")["headcount_status"].value_counts()

PASS employees_laid_off: all filled values are whole numbers
PASS total_employees: all filled values are whole numbers
PASS severance_weeks: all filled values are whole numbers
PASS news_mentions: all filled values are whole numbers


source_dataset          headcount_status   
fliar_global_sample     verified               2489
                        failed_check           1314
                        not_disclosed          1197
tech_layoffs_2025_2026  reported_unverified      35
Name: count, dtype: int64

In [22]:
check("A-08: missing headcounts stay missing (2,511 global, 0 tech)",
      events.loc[~is_tech, "employees_laid_off"].isna().sum() == raw_global["headcount_cut"].isna().sum() == 2511
      and events.loc[is_tech, "employees_laid_off"].isna().sum() == 0)
check("no zero headcount was created", (events["employees_laid_off"] == 0).sum() == 0)
check("A-09: global total_employees all missing", events.loc[~is_tech, "total_employees"].isna().all())
check("workforce_pct missing count unchanged (3,879)", events["workforce_pct"].isna().sum() == 3879)

PASS A-08: missing headcounts stay missing (2,511 global, 0 tech)
PASS no zero headcount was created
PASS A-09: global total_employees all missing
PASS workforce_pct missing count unchanged (3,879)


## 10. Step 7: derived reason, AI mention and permanent-layoff flag (A-10, A-12, A-16)

**Objective.** Add three *derived* columns, clearly labelled as derived, without changing the source fields.

**Concept.**
- **A-10:** `reason` keeps the source text (tech only). `reason_category_derived` is a keyword category from published rules (`src/phase2_cleaning.py`, `REASON_RULES`).
  It reads the news excerpt for global rows and the source reason for tech rows. `reason_method` says which.
- **A-12:** `ai_related` is the tech file's own Yes/Partial/No (missing for global). `ai_mentioned_in_news_derived` is a separate keyword flag for global rows.
- **A-16:** `is_permanent_layoff` = not a furlough (global). Unknown (missing) for tech, because the tech file has no furlough field.

In [23]:
reason_text = np.where(is_tech, events["reason"], events["news_excerpt"])
events["reason_category_derived"] = [pc.derive_reason(x if isinstance(x, str) else None) for x in reason_text]
events["reason_method"] = np.where(is_tech, "keyword_rule_on_source_reason", "keyword_rule_on_news_excerpt")
change_log.append({"rule_id": "T08", "decision": "A-10", "column": "reason_category_derived",
                   "description": "new derived column from keyword rules; source 'reason' unchanged",
                   "rows_changed": int((events["reason_category_derived"] != "Not stated").sum()), "examples": ""})

events["ai_mentioned_in_news_derived"] = pd.Series(pd.NA, index=events.index, dtype="boolean")
events.loc[~is_tech, "ai_mentioned_in_news_derived"] = pc.ai_mentioned(events.loc[~is_tech, "news_excerpt"])
change_log.append({"rule_id": "T09", "decision": "A-12", "column": "ai_mentioned_in_news_derived",
                   "description": "new derived flag (global only); ai_related unchanged",
                   "rows_changed": int(events["ai_mentioned_in_news_derived"].sum()), "examples": ""})

events["is_furlough"] = events["is_furlough"].astype("boolean")
events["is_permanent_layoff"] = ~events["is_furlough"]          # NA stays NA for tech rows
change_log.append({"rule_id": "T10", "decision": "A-16", "column": "is_permanent_layoff",
                   "description": "NOT is_furlough; missing for tech", "rows_changed": int((~is_tech).sum()), "examples": ""})

pd.crosstab(events["reason_category_derived"], events["source_dataset"])

source_dataset,fliar_global_sample,tech_layoffs_2025_2026
reason_category_derived,,
AI / Automation,43,9
Closure / Bankruptcy,444,0
Cost Cutting,220,11
Demand / Market,247,7
Merger / Acquisition,54,0
Not stated,3509,2
Offshoring / Relocation,26,0
Pandemic,174,1
Restructuring,283,5


In [24]:
check("A-10: global reason stays missing", events.loc[~is_tech, "reason"].isna().all())
check("A-12: ai_related missing for global, unchanged for tech", events.loc[~is_tech, "ai_related"].isna().all()
      and events.loc[is_tech, "ai_related"].tolist() == raw_tech["AI_Related"].tolist())
check("A-12: AI flag missing where the excerpt is missing",
      events.loc[~is_tech & events["news_excerpt"].isna(), "ai_mentioned_in_news_derived"].isna().all())
check("A-16: furlough rows kept (133) and permanent flag missing for tech",
      events["is_furlough"].sum() == 133 and events.loc[is_tech, "is_permanent_layoff"].isna().all())
print("AI mentioned in global news excerpts:", int(events["ai_mentioned_in_news_derived"].sum()))

PASS A-10: global reason stays missing
PASS A-12: ai_related missing for global, unchanged for tech
PASS A-12: AI flag missing where the excerpt is missing
PASS A-16: furlough rows kept (133) and permanent flag missing for tech
AI mentioned in global news excerpts: 41


## 11. Step 8: tidy news URLs (A-14)

**Objective.** Remove the line breaks and spaces around 6 news URLs, in a new `news_url` column. `news_url_raw` keeps the original.

In [25]:
events["news_url"] = events["news_url_raw"].astype("string").str.strip()
n = pc.log_change(change_log, "T11", "A-14", "news_url", events["news_url_raw"], events["news_url"],
                  "leading/trailing whitespace removed", events["event_key"])
check("A-14: exactly the 6 padded URLs changed", n == 6)

PASS A-14: exactly the 6 padded URLs changed


## 12. Step 9: link possible duplicates, never merge (A-13, A-15)

**Objective.** Mark which records belong together as *possible* duplicates and keep both review files exactly as they are.

**Concept.** Records that appear together in one of the **18 kept pairs** share a `possible_duplicate_group` (e.g. Intel's tech row and its 4 nearby global rows).
This is a **link**, not a merge: all rows stay, and no record is chosen as "primary" until you review the pairs.
The 97 *excluded* pairs are not used for groups (they failed the matching rule), but their file is preserved.
**A-15:** the 42 global rows that share a news URL with the same company get `possible_internal_duplicate = True`.

In [26]:
groups = pc.duplicate_groups(kept_pairs)
events["possible_duplicate_group"] = events["event_key"].map(groups).astype("string")
events["possible_internal_duplicate"] = (events["event_key"]
                                         .map(staged_global.set_index("record_id")["qc_same_company_same_url"])
                                         .astype("boolean"))
# tech rows have no news URLs, so this check was never computed for them: they stay missing (NA), not False
change_log.append({"rule_id": "T12", "decision": "A-13", "column": "possible_duplicate_group",
                   "description": "link records in the 18 kept pairs (connected groups); nothing merged",
                   "rows_changed": int(events["possible_duplicate_group"].notna().sum()), "examples": ""})
change_log.append({"rule_id": "T13", "decision": "A-15", "column": "possible_internal_duplicate",
                   "description": "Phase 1 same-company shared-URL flag carried over",
                   "rows_changed": int(events["possible_internal_duplicate"].sum()), "examples": ""})
events.groupby("possible_duplicate_group").agg(rows=("event_key", "count"), companies=("company_name", lambda s: ", ".join(sorted(set(s)))))

,rows,companies
possible_duplicate_group,,
DG-001,3,Accenture
DG-002,2,Adobe
DG-003,2,Dell
DG-004,2,Roku
DG-005,2,Twitch
DG-006,3,Unity
DG-007,2,Cisco
DG-008,5,Intel
DG-009,4,Microsoft


In [27]:
for name in ["possible_duplicate_pairs.csv", "excluded_candidate_pairs.csv"]:
    shutil.copyfile(INTERIM_DIR / name, PROCESSED_DIR / name)                   # byte-for-byte copy
    check(f"{name} copied unchanged", sha256_of(INTERIM_DIR / name) == sha256_of(PROCESSED_DIR / name))
check("A-13: 11 groups covering 29 records (11 tech + 18 global)",
      events["possible_duplicate_group"].nunique() == 11 and events["possible_duplicate_group"].notna().sum() == 29)
check("A-15: 42 internal-duplicate rows; not computed (NA) for tech",
      events["possible_internal_duplicate"].sum() == 42 and events.loc[is_tech, "possible_internal_duplicate"].isna().all())

PASS possible_duplicate_pairs.csv copied unchanged
PASS excluded_candidate_pairs.csv copied unchanged
PASS A-13: 11 groups covering 29 records (11 tech + 18 global)
PASS A-15: 42 internal-duplicate rows; not computed (NA) for tech


## 13. Step 10: quality flags and the quality-issues report

**Objective.** Bring every Phase 1 `qc_` flag across unchanged, add the Phase 2 flags, and write one long table of all issues.

**Concept.** A flag that was never computed for a source is left **missing (NA)**, not `False`. For example, the headcount/status conflict check only exists for global rows.
`qc_issue_count` counts the `True` flags per row. `quality_issues.csv` has one row per (record, flag), which is easy to filter in Power BI or SQL.

In [28]:
staged_flags = pd.concat([staged_global.rename(columns={"record_id": "event_key"}),
                          staged_tech.rename(columns={"record_id": "event_key"})], ignore_index=True)
for col in pc.PHASE1_QC_COLUMNS:
    source_values = staged_flags.set_index("event_key")[col] if col in staged_flags else pd.Series(dtype="boolean")
    events[col] = events["event_key"].map(source_values).astype("boolean")   # NA where not computed for that source

events["qc_suspect_date"] = is_suspect
events["qc_company_name_needs_review"] = events["company_name_review_status"] == "needs_review"
for col in pc.QC_COLUMNS:
    events[col] = events[col].astype("boolean")
events["qc_issue_count"] = events[pc.QC_COLUMNS].fillna(False).sum(axis=1).astype("Int64")

flag_summary = pd.DataFrame({"global_true": events.loc[~is_tech, pc.QC_COLUMNS].sum(),
                             "tech_true": events.loc[is_tech, pc.QC_COLUMNS].sum(),
                             "global_not_computed": events.loc[~is_tech, pc.QC_COLUMNS].isna().sum(),
                             "tech_not_computed": events.loc[is_tech, pc.QC_COLUMNS].isna().sum()})
flag_summary

,global_true,tech_true,global_not_computed,tech_not_computed
qc_date_unparseable,0,0,0,0
qc_date_outside_expected_range,0,0,0,0
qc_date_after_audit_date,0,0,0,0
qc_non_numeric_value,0,0,0,0
qc_headcount_not_positive,0,0,0,0
qc_pct_out_of_range,0,0,0,0
qc_headcount_status_conflict,0,0,0,35
qc_headcount_not_verified,2511,0,0,35
qc_missing_sector,1564,0,0,35
qc_missing_country,470,0,0,35


In [29]:
# Validation: Phase 1 flags came across unchanged
for col in pc.PHASE1_QC_COLUMNS:
    if col in staged_global:
        same = events.loc[~is_tech, col].astype(bool).tolist() == staged_global[col].astype(bool).tolist()
        check(f"Phase 1 flag carried unchanged: {col} (global)", same)

long = events.melt(id_vars=["event_key", "source_dataset", "source_row_number", "company_name"],
                   value_vars=pc.QC_COLUMNS, var_name="quality_flag", value_name="flag_value")
quality_issues = long[long["flag_value"] == True].drop(columns="flag_value")   # noqa: E712  (NA is not an issue)
quality_issues = quality_issues.sort_values(["event_key", "quality_flag"]).reset_index(drop=True)
check("quality_issues rows = total True flags", len(quality_issues) == int(events["qc_issue_count"].sum()),
      f"{len(quality_issues)} issues")

PASS Phase 1 flag carried unchanged: qc_date_unparseable (global)
PASS Phase 1 flag carried unchanged: qc_date_outside_expected_range (global)
PASS Phase 1 flag carried unchanged: qc_date_after_audit_date (global)
PASS Phase 1 flag carried unchanged: qc_non_numeric_value (global)
PASS Phase 1 flag carried unchanged: qc_headcount_not_positive (global)
PASS Phase 1 flag carried unchanged: qc_pct_out_of_range (global)
PASS Phase 1 flag carried unchanged: qc_headcount_status_conflict (global)
PASS Phase 1 flag carried unchanged: qc_headcount_not_verified (global)
PASS Phase 1 flag carried unchanged: qc_missing_sector (global)
PASS Phase 1 flag carried unchanged: qc_missing_country (global)
PASS Phase 1 flag carried unchanged: qc_noisy_company_name (global)
PASS Phase 1 flag carried unchanged: qc_same_company_same_url (global)
PASS Phase 1 flag carried unchanged: qc_possible_cross_source_dup (global)
PASS quality_issues rows = total True flags  (4886 issues)


## 14. Step 11: department bridge table (A-11)

**Objective.** Split tech departments such as `"Engineering, Marketing"` into one row per department, without allowing double counting.

**Concept: bridge table.** One layoff event can hit two departments. If we copied the headcount onto both department rows, a sum would count the layoff twice.
So the bridge has **no headcount column**. To count *events* by department use `COUNT(DISTINCT event_key)`, and to split one event across its departments use `allocation_weight` (they add up to 1 per event).

In [30]:
departments = pc.departments_bridge(events)
check("bridge: every event_key exists in layoff_events", departments["event_key"].isin(events["event_key"]).all())
check("bridge: (event_key, department) is unique", not departments.duplicated(["event_key", "department"]).any())
check("bridge: weights add to 1 per event", np.allclose(departments.groupby("event_key")["allocation_weight"].sum(), 1.0))
check("bridge: covers all 35 tech events and only them",
      set(departments["event_key"]) == set(events.loc[events["department"].notna(), "event_key"])
      and departments["event_key"].nunique() == 35)
check("bridge: no headcount column (cannot double count)",
      not {"employees_laid_off", "headcount"} & set(departments.columns))
check("bridge: source_row_number matches layoff_events",
      (departments["source_row_number"] == departments["event_key"].map(events.set_index("event_key")["source_row_number"])).all())
print(len(departments), "bridge rows")
departments.head(6)

PASS bridge: every event_key exists in layoff_events
PASS bridge: (event_key, department) is unique
PASS bridge: weights add to 1 per event
PASS bridge: covers all 35 tech events and only them
PASS bridge: no headcount column (cannot double count)
PASS bridge: source_row_number matches layoff_events
70 bridge rows


,event_key,source_dataset,source_row_number,department_raw,department,department_position,departments_in_event,allocation_weight
0,T-00001,tech_layoffs_2025_2026,2,"Content, Marketing",Content,1,2,0.5
1,T-00001,tech_layoffs_2025_2026,2,"Content, Marketing",Marketing,2,2,0.5
2,T-00002,tech_layoffs_2025_2026,3,"Engineering, Marketing",Engineering,1,2,0.5
3,T-00002,tech_layoffs_2025_2026,3,"Engineering, Marketing",Marketing,2,2,0.5
4,T-00003,tech_layoffs_2025_2026,4,"Engineering, Marketing",Engineering,1,2,0.5
5,T-00003,tech_layoffs_2025_2026,4,"Engineering, Marketing",Marketing,2,2,0.5


## 15. Final validation: schema, keys, row counts, lineage and missingness

**Objective.** Prove the canonical table is complete, correctly shaped and traceable before saving it.

**Concept: lineage.** For every row we go back to the exact raw CSV line (`source_row_number`) and compare the original values. If even one differs, the notebook stops.

In [31]:
layoff_events = events[pc.CANONICAL_COLUMNS + pc.QC_COLUMNS].copy()

check("schema: exact column list", list(layoff_events.columns) == pc.CANONICAL_COLUMNS + pc.QC_COLUMNS,
      f"{len(pc.CANONICAL_COLUMNS)} named + {len(pc.QC_COLUMNS)} qc = {layoff_events.shape[1]} columns")
check("row count = 5,000 global + 35 tech", (layoff_events["source_dataset"].value_counts().to_dict()
                                             == {pc.GLOBAL_NAME: 5000, pc.TECH_NAME: 35}))
check("primary key event_key unique and never missing",
      layoff_events["event_key"].is_unique and layoff_events["event_key"].notna().all())
check("(source_dataset, source_row_number) unique",
      not layoff_events.duplicated(["source_dataset", "source_row_number"]).any())
check("global source_event_id = layoff_id, unique", layoff_events.loc[~is_tech, "source_event_id"].is_unique
      and layoff_events.loc[~is_tech, "source_event_id"].tolist() == raw_global["layoff_id"].tolist())
check("A-01: tech source_event_id missing", layoff_events.loc[is_tech, "source_event_id"].isna().all())
check("TECH: confidence high for global, low for tech",
      (layoff_events.loc[~is_tech, "source_confidence"] == "high").all() and (layoff_events.loc[is_tech, "source_confidence"] == "low").all())

PASS schema: exact column list  (51 named + 18 qc = 69 columns)
PASS row count = 5,000 global + 35 tech
PASS primary key event_key unique and never missing
PASS (source_dataset, source_row_number) unique
PASS global source_event_id = layoff_id, unique
PASS A-01: tech source_event_id missing
PASS TECH: confidence high for global, low for tech


In [32]:
# Lineage: every raw-preserving column equals the raw file at its source line
def raw_value_matches(ds_mask, raw, mapping):
    rows = layoff_events[ds_mask].sort_values("source_row_number")
    back = raw.iloc[rows["source_row_number"].to_numpy() - 2].reset_index(drop=True)
    bad = []
    for canonical, source_col in mapping.items():
        if source_col is None or canonical in ("employees_laid_off", "workforce_pct", "total_employees",
                                               "severance_weeks", "news_mentions", "is_furlough", "headcount_status"):
            continue                                                     # typed/relabelled columns checked below
        a = rows[canonical].astype("string").reset_index(drop=True)
        b = back[source_col].astype("string")
        if not a.fillna("<NA>").equals(b.fillna("<NA>")):
            bad.append(canonical)
    return bad


bad_g = raw_value_matches(~is_tech, raw_global, GLOBAL_TO_CANONICAL)
bad_t = raw_value_matches(is_tech, raw_tech, TECH_TO_CANONICAL)
check("lineage: raw-preserving columns equal raw values at source line (global)", not bad_g, bad_g)
check("lineage: raw-preserving columns equal raw values at source line (tech)", not bad_t, bad_t)
num_ok = (pd.to_numeric(raw_global["headcount_cut"]).astype("Int64").equals(layoff_events.loc[~is_tech, "employees_laid_off"].reset_index(drop=True))
          and raw_tech["Employees_Laid_Off"].astype("Int64").equals(layoff_events.loc[is_tech, "employees_laid_off"].reset_index(drop=True)))
check("lineage: numeric headcounts equal raw values after type change", num_ok)

PASS lineage: raw-preserving columns equal raw values at source line (global)
PASS lineage: raw-preserving columns equal raw values at source line (tech)
PASS lineage: numeric headcounts equal raw values after type change


In [33]:
# Missingness: raw vs canonical, field by field (must be identical unless a rule says otherwise)
pairs_to_compare = [("employees_laid_off", "headcount_cut", "Employees_Laid_Off"), ("workforce_pct", "pct_workforce", "Percentage_Workforce"),
                    ("industry_raw", "sector", "Industry"), ("country_raw", "country", "Country"),
                    ("hq_country", "company_country", None), ("company_domain", "company_domain", None),
                    ("news_url", "news_url", None), ("news_excerpt", "news_excerpt", None)]
miss = []
for canonical, gcol, tcol in pairs_to_compare:
    raw_missing = raw_global[gcol].isna().sum() + (raw_tech[tcol].isna().sum() if tcol else len(raw_tech))
    miss.append({"column": canonical, "missing_raw": int(raw_missing), "missing_canonical": int(layoff_events[canonical].isna().sum())})
miss = pd.DataFrame(miss)
miss["same"] = miss["missing_raw"] == miss["missing_canonical"]
check("missingness preserved for every compared field", miss["same"].all())
miss

PASS missingness preserved for every compared field


,column,missing_raw,missing_canonical,same
0,employees_laid_off,2511,2511,True
1,workforce_pct,3879,3879,True
2,industry_raw,1564,1564,True
3,country_raw,470,470,True
4,hq_country,1680,1680,True
5,company_domain,1300,1300,True
6,news_url,1646,1646,True
7,news_excerpt,45,45,True


## 16. Save outputs, fingerprint them and update the audit history

**Objective.** Write every output to `data/processed/`, record SHA-256 fingerprints so a re-run can be compared, and confirm the raw files are still unchanged.

**Concept: reproducibility.** The canonical table contains no timestamps or random values, so running this notebook again on the same inputs must give the **same fingerprint**.
`phase2_output_manifest.csv` stores the fingerprints, and the tests compare a fresh run against them.

In [34]:
company_map = (events.groupby(["source_dataset", "company_name_raw", "company_name", "company_name_method",
                               "company_name_review_status"]).size().rename("rows").reset_index())
industry_map = (events.assign(industry_raw=events["industry_raw"].fillna("(missing)"))
                .groupby(["source_dataset", "industry_raw", "industry_group"]).size().rename("rows").reset_index())
lineage = pd.DataFrame(
    [{"canonical_field": c, "source_dataset": pc.GLOBAL_NAME, "source_field": s or "(none: missing)"} for c, s in GLOBAL_TO_CANONICAL.items()] +
    [{"canonical_field": c, "source_dataset": pc.TECH_NAME, "source_field": s or "(none: missing)"} for c, s in TECH_TO_CANONICAL.items()] +
    [{"canonical_field": "(dropped)", "source_dataset": ds, "source_field": c} for ds, cols in
     [(pc.GLOBAL_NAME, DROPPED["global"]), (pc.TECH_NAME, DROPPED["tech"])] for c in cols])
transformation_log = pd.DataFrame(change_log)

outputs = {
    "layoff_events.csv": layoff_events, "layoff_event_departments.csv": departments,
    "quality_issues.csv": quality_issues, "transformation_log.csv": transformation_log,
    "field_lineage.csv": lineage, "company_name_mapping.csv": company_map, "industry_mapping.csv": industry_map,
}
for name, df in outputs.items():
    df.to_csv(PROCESSED_DIR / name, index=False, date_format="%Y-%m-%d")

files = sorted(list(outputs) + ["cross_source_industry_disagreements.csv", "possible_duplicate_pairs.csv", "excluded_candidate_pairs.csv"])
out_manifest = pd.DataFrame({"file": files,
                             "rows": [len(pd.read_csv(PROCESSED_DIR / f)) for f in files],
                             "sha256": [sha256_of(PROCESSED_DIR / f) for f in files]})
out_manifest.to_csv(PROCESSED_DIR / "phase2_output_manifest.csv", index=False)
out_manifest

,file,rows,sha256
0,company_name_mapping.csv,3532,54be3d99e71abbb7db7fba58dd0bc661d0d9a4f9ad562c...
1,cross_source_industry_disagreements.csv,9,5a375a3ffa7eb88d7311c99bab906105dfcfbd69a2ae3a...
2,excluded_candidate_pairs.csv,97,97e9fd91b9262cdce22c6764d52bfabebd88b13a8d27a3...
3,field_lineage.csv,54,3b84aeec32c03867ae4abb24cea4a22423d1830a49bd50...
4,industry_mapping.csv,77,1c1338df08101114f974e466e39209e4c4db387c33d32a...
5,layoff_event_departments.csv,70,08005235def78f8ecfed9b60a0354e32f2af93f96af6c3...
6,layoff_events.csv,5035,9a42f4f1a1a7a4887c245e72abb64b1cee1d8be7f0ed95...
7,possible_duplicate_pairs.csv,18,8a8379fe5fb3da2a4e3ab155280afda353b066c6a10faa...
8,quality_issues.csv,4886,3159195d3bb7404600fcf804c35a5d1a4c723a34152e36...
9,transformation_log.csv,16,365beee7cdc663572cc1449ae209226684c5bc0a00cb4b...


In [35]:
# Reload check: the saved canonical table reads back with the same shape and key
reloaded = pd.read_csv(PROCESSED_DIR / "layoff_events.csv", keep_default_na=False, na_values=[""])
check("saved layoff_events reloads with 5,035 rows and same columns",
      reloaded.shape == layoff_events.shape and list(reloaded.columns) == list(layoff_events.columns))
check("saved file keeps 2,511 missing global headcounts (no zeros)",
      reloaded.loc[reloaded["source_dataset"] == pc.GLOBAL_NAME, "employees_laid_off"].isna().sum() == 2511
      and (reloaded["employees_laid_off"] == 0).sum() == 0)

manifest_after = [sha256_of(RAW_DIR / f) for f in manifest["file"]]
check("raw files still unchanged at the end", manifest_after == manifest["sha256"].tolist())

run_id = datetime.now(timezone.utc).strftime("phase2_run_%Y%m%dT%H%M%SZ")
checks = pd.DataFrame(check_log).assign(run_id=run_id)
checks.to_csv(AUDIT_DIR / "phase2_check_log.csv", index=False)
history = AUDIT_DIR / "audit_log_history.csv"
(checks.rename(columns={"passed": "issues_found"})
       .assign(issues_found=lambda d: (~d["issues_found"]).astype(int), dataset="phase2", records_checked=len(layoff_events),
               action_taken="validation check", notes=checks["detail"])
       [["checked_at_utc", "check", "dataset", "records_checked", "issues_found", "action_taken", "notes", "run_id"]]
       .to_csv(history, mode="a", header=not history.exists(), index=False))
print(f"{len(checks)} checks, all passed; appended to audit history as {run_id}")

PASS saved layoff_events reloads with 5,035 rows and same columns
PASS saved file keeps 2,511 missing global headcounts (no zeros)
PASS raw files still unchanged at the end
75 checks, all passed; appended to audit history as phase2_run_20261023T101736Z


In [36]:
transformation_log[["rule_id", "decision", "column", "rows_changed", "description"]]

,rule_id,decision,column,rows_changed,description
0,T01,A-02/P2,company_name,224,legal-suffix/spacing rule + confident manual m...
1,T01b,A-02,company_name,7,spellings differing only by capital letters: p...
2,T02,mechanical,event_date,0,text parsed to date (format unchanged)
3,T03,A-07,date_reliability,6,flag only: tech rows matching a global event's...
4,T04,A-04/P1,industry_group,4772,label-level mapping to 16 groups + Unknown (mi...
5,T05,A-03,country,36,4 approved relabels + trim
6,T05b,A-03/A-05,hq_country,6,4 approved relabels + trim; never filled from ...
7,T06,A-06,city,35,tech: text before the comma
8,T06b,A-06,state,31,tech: after the comma only if a US state code
9,T07,A-08,headcount_status,35,tech rows labelled reported_unverified


## 17. Summary

- `layoff_events.csv`: **5,035 rows** (5,000 global + 35 tech), one per source record, original values next to cleaned values, all flags carried.
- Every transformation is in `transformation_log.csv`, every issue in `quality_issues.csv`, and every column's origin in `field_lineage.csv`.
- The 6 suspect tech dates are **flagged, not changed**. The 13 uncertain company names keep their raw name. Industry disagreements between sources are **recorded, not overridden**.
- Both duplicate-review files are copied byte-for-byte. 29 records are linked into 11 possible-duplicate groups and **nothing is merged**.
- Still open for your review: the 18 + 97 candidate pairs, 13 company names, and the non-country values flagged in Step 5.

**Next phase (only after you verify this one):** exploratory analysis (notebook 03).